# Elise Crash Analysis

Find and inspect the worst greedy evaluation episodes for a saved SolarSystemLander checkpoint.

In [ ]:
# cell: colab-setup

!git clone https://github.com/miwehle/rl_lab.git
%cd rl_lab
%pip install -r hpo/requirements.txt

import sys

sys.path.insert(0, "dqn/src")
sys.path.insert(0, "hpo/src")

In [ ]:
# cell: crash-analysis-setup; requires: colab-setup

import torch

from dqn.model import DQN
from hpo.evaluation.video import InfraCfg, checkpoint_metadata
from hpo.objective import ObjectiveConfig
from hpo.environments.solar_system_lander.env import EnvFactory, World

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OBSERVATION_MODE = "10d"
DATABASE = f"solar_system_lander_{OBSERVATION_MODE}_elise_stp"

WORLD_MIX = {World.MERCURY: 1, World.VENUS: 4, World.EARTH: 4, World.MOON: 1, World.MARS: 1}
ENV_FACTORY = EnvFactory(OBSERVATION_MODE, world_mix=WORLD_MIX)

CHECKPOINT_PATH = InfraCfg().checkpoint_path(DATABASE)

OBJECTIVE_CFG = ObjectiveConfig(
    environment_factory=ENV_FACTORY,
    num_envs=22,
    eval_episodes=10,
    device=device,
)

metadata = checkpoint_metadata(DATABASE)
print(f"checkpoint: {CHECKPOINT_PATH}")
print(f"source score: {metadata['score']:.1f}")
print(f"trial: {metadata['trial_number']}")
metadata["world_scores"]

In [ ]:
# cell: worst-checkpoint-episodes; requires: crash-analysis-setup

from hpo.evaluation.checkpoint_robustness import checkpoint_scores, score_summary

_EPISODES = 200
_WORST_N = 20

scores = checkpoint_scores(CHECKPOINT_PATH, OBJECTIVE_CFG, episodes=_EPISODES)

if OBJECTIVE_CFG.eval_seed is None:
    scores["seed"] = None
else:
    scores["seed"] = scores["episode"].astype(int) + OBJECTIVE_CFG.eval_seed

worst_checkpoint_episodes = scores.sort_values("score").head(_WORST_N).reset_index(drop=True)

display(score_summary(scores).round(1))
display(worst_checkpoint_episodes)

In [ ]:
# cell: worst-video-jobs; requires: worst-checkpoint-episodes

import pandas as pd

from hpo.evaluation.video import video_conditions_table

_VIDEO_N = 5

worst_video_jobs = worst_checkpoint_episodes.head(_VIDEO_N).copy()

_condition_rows = []
for nr, row in enumerate(worst_video_jobs.itertuples(index=False)):
    _table = video_conditions_table(ENV_FACTORY, [row.world], [int(row.seed)]).drop(columns=["nr"])
    _table.insert(0, "nr", nr)
    _table.insert(3, "score", row.score)
    _condition_rows.append(_table)

worst_video_conditions = pd.concat(_condition_rows, ignore_index=True)
display(worst_video_conditions)

In [ ]:
# cell: action-channel-analysis; requires: worst-video-jobs

from collections import Counter

from hpo.evaluation.checkpoint_robustness import q_net_from_checkpoint

_ACTION_NAMES = {0: "noop", 1: "left", 2: "main", 3: "right"}

_q_net_world = str(worst_video_jobs.iloc[0]["world"])
_q_net = q_net_from_checkpoint(
    CHECKPOINT_PATH,
    make_env=lambda: ENV_FACTORY.make_env(_q_net_world),
    device=device,
)
_q_net.eval()


def _greedy_action(observation):
    _state = torch.as_tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)
    with torch.no_grad():
        return int(_q_net(_state).argmax(dim=1).item())


def _analyze_episode(world, seed, expected_score):
    _env = ENV_FACTORY.make_env(str(world))
    _counts = Counter()
    _score = 0.0
    _steps = 0
    _max_sink_obs_vy = 0.0
    _max_abs_angle = 0.0
    _max_abs_angular_velocity = 0.0
    _first_contact_step = None
    _terminated = False
    _truncated = False
    try:
        _observation, _ = _env.reset(seed=int(seed))
        for _step in range(OBJECTIVE_CFG.eval_max_steps):
            _max_sink_obs_vy = max(_max_sink_obs_vy, max(0.0, -float(_observation[3])))
            _max_abs_angle = max(_max_abs_angle, abs(float(_observation[4])))
            _max_abs_angular_velocity = max(
                _max_abs_angular_velocity, abs(float(_observation[5]))
            )

            _action = _greedy_action(_observation)
            _counts[_action] += 1
            _observation, _reward, _terminated, _truncated, _ = _env.step(_action)
            _score += float(_reward)
            _steps = _step + 1

            if _first_contact_step is None and any(
                getattr(_leg, "ground_contact", False)
                for _leg in getattr(_env.unwrapped, "legs", ())
            ):
                _first_contact_step = _steps
            if _terminated or _truncated:
                break
    finally:
        _env.close()

    _side_count = _counts[1] + _counts[3]
    _main_count = _counts[2]
    return {
        "world": str(world),
        "seed": int(seed),
        "expected_score": float(expected_score),
        "repro_score": _score,
        "score_delta": _score - float(expected_score),
        "steps": _steps,
        "first_contact_step": _first_contact_step,
        "terminated": _terminated,
        "truncated": _truncated,
        "noop_count": _counts[0],
        "left_count": _counts[1],
        "main_count": _main_count,
        "right_count": _counts[3],
        "side_count": _side_count,
        "noop_fraction": _counts[0] / _steps,
        "left_fraction": _counts[1] / _steps,
        "main_fraction": _main_count / _steps,
        "right_fraction": _counts[3] / _steps,
        "side_fraction": _side_count / _steps,
        "main_every_n_steps": None if _main_count == 0 else _steps / _main_count,
        "max_sink_obs_vy": _max_sink_obs_vy,
        "final_obs_vy": float(_observation[3]),
        "max_abs_angle": _max_abs_angle,
        "final_angle": float(_observation[4]),
        "max_abs_angular_velocity": _max_abs_angular_velocity,
        "final_angular_velocity": float(_observation[5]),
    }


action_channel_analysis = pd.DataFrame(
    _analyze_episode(row.world, row.seed, row.score)
    for row in worst_video_jobs.itertuples(index=False)
)
display(action_channel_analysis.round(3))

In [ ]:
# cell: record-worst-videos; requires: worst-video-jobs

from hpo.evaluation.rendering.solar_system_lander import render_config
from hpo.evaluation.video import record_video

worst_video_paths = [
    record_video(
        DQN,
        ENV_FACTORY.make_env(str(row.world), render_mode="rgb_array"),
        study_name=DATABASE,
        seed=int(row.seed),
        device=device,
        max_steps=OBJECTIVE_CFG.eval_max_steps,
        render_cfg=render_config([row.world], overlay=True),
    )
    for row in worst_video_jobs.itertuples(index=False)
]

worst_video_paths

In [ ]:
# cell: display-worst-video; requires: record-worst-videos

from hpo.evaluation.video import display_video

display_video(worst_video_paths, 0)